# Catalog Aggregation Verification (Tick-to-Candle)

Validates that every catalog-stored aggregated bar (5-MIN through 1-MONTH × EURUSD / GBPUSD / USDJPY × BID / ASK = **48 tuples**) is a correct OHLCV roll-up of the underlying `1-MINUTE-...-EXTERNAL` source bars.

- Catalog: `catalog/data/bar/`
- Source bar type: 1-MINUTE `-EXTERNAL` (loaded directly from CSV)
- Aggregated bar type: `-INTERNAL` (produced by the in-process aggregator pipeline)
- Price tolerance: `1e-08` | Volume tolerance: `1e-06`
- Bucketing rule: window `(T - N, T]` stamped at `T` (pandas `closed='right', label='right'`); volume clipped to `QUANTITY_MAX`.

## OHLCV invariant

Each aggregated bar is reconstructed from its 1-MIN constituents as:

- **O** = first `open` in the window
- **H** = max `high` in the window
- **L** = min `low` in the window
- **C** = last `close` in the window
- **V** = sum of `volume` in the window (clipped to `QUANTITY_MAX`)

Three-way agreement is required across:
1. **Path A** — `core.aggregator.aggregate_ohlcv` (production logic)
2. **Path B** — an independent pandas resample mirroring the same rule
3. **Cat**  — the bars stored in the parquet catalog


In [1]:
import gc
import gzip
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Windows: force stdout to UTF-8 so any glyphs in printed tables don't blow up.
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass


def _find_project_root() -> Path:
    marker = Path("core") / "instrument_factory.py"

    def _walk_up(p: Path) -> Path | None:
        for parent in (p, *p.parents):
            if (parent / marker).is_file():
                return parent
        return None

    found = _walk_up(Path.cwd())
    if found:
        return found
    for var in ("__vsc_ipynb_file__", "__session__", "__file__"):
        nb = globals().get(var)
        if nb:
            found = _walk_up(Path(nb).resolve().parent)
            if found:
                return found
    for child in Path.cwd().iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    raise RuntimeError(
        f"could not find project root (no `{marker}`) from cwd={Path.cwd()}"
    )


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

INSTRUMENTS = ("EURUSD", "GBPUSD", "USDJPY")
SIDES = ("ASK", "BID")
TIMEFRAMES = (
    "5-MINUTE",
    "15-MINUTE",
    "30-MINUTE",
    "1-HOUR",
    "2-HOUR",
    "1-DAY",
    "1-WEEK",
    "1-MONTH",
)
TIMEFRAME_RULE_MIN = {
    "5-MINUTE": 5,
    "15-MINUTE": 15,
    "30-MINUTE": 30,
    "1-HOUR": 60,
    "2-HOUR": 120,
    "1-DAY": 24 * 60,
}

SRC_SUFFIX = "EXTERNAL"
AGG_SUFFIX = "INTERNAL"
PRICE_TOL = 1e-8
VOLUME_TOL = 1e-6

WINDOWS_CSV = REPORT_DIR / "catalog_aggregation_windows.csv.gz"
WINDOWS_SAMPLE_CSV = REPORT_DIR / "catalog_aggregation_windows_sample.csv"

print(f"project root : {PROJECT_ROOT}")
print(f"report dir   : {REPORT_DIR}")
print(f"primary CSV  : {WINDOWS_CSV.name}")
print(f"sample  CSV  : {WINDOWS_SAMPLE_CSV.name}")

project root : c:\Users\HP\OneDrive\Desktop\m-cube_version1
report dir   : c:\Users\HP\OneDrive\Desktop\m-cube_version1\reports
primary CSV  : catalog_aggregation_windows.csv.gz
sample  CSV  : catalog_aggregation_windows_sample.csv


In [2]:
from core.aggregator import OHLCV_AGG, _rule_for, aggregate_ohlcv
from core.csv_loader import QUANTITY_MAX
from verify_final import DATA_DIR, load_bars

print(f"catalog data dir : {DATA_DIR}")
print(f"QUANTITY_MAX     : {QUANTITY_MAX:,.0f}")

catalog data dir : C:\Users\HP\OneDrive\Desktop\m-cube_version1\catalog\data\bar
QUANTITY_MAX     : 16,000,000,000


## Helpers — bar-type strings and frame comparators

`bar_type` builds the Nautilus bar-type string used as the catalog dir name. `manual_resample` is Path B: an independent pandas resample with the same `closed='right', label='right'` rule and the same trailing-partial-drop logic as `aggregate_ohlcv`. `compare_frames` aligns two OHLCV DataFrames on the union of their timestamps and counts how many bars match within tolerance (and which ones don't).

In [3]:
def bar_type(instr: str, tf: str, side: str, suffix: str) -> str:
    return f"{instr}.FOREX_MS-{tf}-{side}-{suffix}"


def manual_resample(df_1min: pd.DataFrame, timeframe: str) -> pd.DataFrame:
    """Path B: independent pandas resample mirroring `aggregate_ohlcv`."""
    if df_1min.empty:
        return df_1min.iloc[0:0].copy()
    rule = _rule_for(timeframe, week_anchor="SUN")
    cols = ["open", "high", "low", "close", "volume"]
    out = (
        df_1min[cols]
        .resample(rule, closed="right", label="right")
        .agg(OHLCV_AGG)
        .dropna(subset=["open", "high", "low", "close"])
    )
    if not out.empty:
        last_t = out.index[-1]
        if not (df_1min.index == last_t).any():
            out = out.iloc[:-1]
    out["volume"] = out["volume"].clip(upper=QUANTITY_MAX)
    return out


def compare_frames(
    expected: pd.DataFrame,
    actual: pd.DataFrame,
    price_tol: float = PRICE_TOL,
    volume_tol: float = VOLUME_TOL,
) -> dict:
    """Bar-by-bar OHLCV comparison on the union of timestamps. Vectorised."""
    if expected.empty and actual.empty:
        return {"n_match": 0, "n_mismatch": 0, "first_mismatches": [], "pass": True}

    union = expected.index.union(actual.index).sort_values()
    exp = expected.reindex(union)
    act = actual.reindex(union)
    price_cols = ("open", "high", "low", "close")

    miss_exp = exp["open"].isna().to_numpy()
    miss_act = act["open"].isna().to_numpy()
    missing = miss_exp | miss_act

    diff = {
        c: np.abs(exp[c].to_numpy(dtype=np.float64) - act[c].to_numpy(dtype=np.float64))
        for c in price_cols
    }
    diff_vol = np.abs(
        exp["volume"].to_numpy(dtype=np.float64) - act["volume"].to_numpy(dtype=np.float64)
    )
    price_bad = np.zeros(len(union), dtype=bool)
    for c in price_cols:
        price_bad |= np.nan_to_num(diff[c], nan=np.inf) > price_tol
    vol_bad = np.nan_to_num(diff_vol, nan=np.inf) > volume_tol

    mismatch_mask = missing | price_bad | vol_bad
    n_mismatch = int(mismatch_mask.sum())
    n_match = len(union) - n_mismatch

    first_mismatches: list[dict] = []
    if n_mismatch:
        bad_idx = np.where(mismatch_mask)[0][:5]
        for i in bad_idx:
            ts = union[i]
            if miss_exp[i] or miss_act[i]:
                first_mismatches.append(
                    {"ts": ts, "reason": "missing in expected" if miss_exp[i] else "missing in actual"}
                )
                continue
            diffs = {}
            for c in price_cols:
                if diff[c][i] > price_tol:
                    diffs[c] = (float(exp[c].iat[i]), float(act[c].iat[i]), float(diff[c][i]))
            if diff_vol[i] > volume_tol:
                diffs["volume"] = (
                    float(exp["volume"].iat[i]),
                    float(act["volume"].iat[i]),
                    float(diff_vol[i]),
                )
            first_mismatches.append({"ts": ts, "diffs": diffs})

    return {
        "n_match": n_match,
        "n_mismatch": n_mismatch,
        "first_mismatches": first_mismatches,
        "pass": n_mismatch == 0,
    }

In [4]:
def _pf(ok: bool) -> str:
    return "PASS" if ok else "FAIL"


def _example_window_start(df_agg: pd.DataFrame, idx: int, timeframe: str) -> pd.Timestamp:
    """Left edge of the (T-N, T] window for bar `idx` of the aggregated frame."""
    agg_ts = df_agg.index[idx]
    if timeframe in TIMEFRAME_RULE_MIN:
        return agg_ts - pd.Timedelta(minutes=TIMEFRAME_RULE_MIN[timeframe])
    if idx > 0:
        return df_agg.index[idx - 1]
    return agg_ts - pd.Timedelta(days=31)


def build_example_bar(
    df_1min: pd.DataFrame, df_catalog: pd.DataFrame, exp_A: pd.DataFrame, timeframe: str
) -> dict:
    """Reconstruct one mid-series aggregated bar from its 1-MIN window."""
    if df_catalog.empty or df_1min.empty or exp_A.empty:
        return {}

    n = len(df_catalog)
    idx = n // 2 if n >= 4 else max(0, n - 1)
    agg_ts = df_catalog.index[idx]
    agg_bar = df_catalog.iloc[idx]

    if agg_ts not in exp_A.index:
        return {}
    a_pos = exp_A.index.get_loc(agg_ts)
    if a_pos > 0:
        window_start = exp_A.index[a_pos - 1]
    else:
        window_start = _example_window_start(df_catalog, idx, timeframe)

    mask = (df_1min.index > window_start) & (df_1min.index <= agg_ts)
    window = df_1min[mask]
    if window.empty:
        return {"agg_ts": agg_ts, "n_1min": 0}

    a_row = exp_A.loc[agg_ts]
    return {
        "agg_ts": agg_ts,
        "window_start": window_start,
        "n_1min": int(len(window)),
        "first_1min_ts": window.index[0],
        "last_1min_ts": window.index[-1],
        "expected": {
            "O": float(a_row["open"]),
            "H": float(a_row["high"]),
            "L": float(a_row["low"]),
            "C": float(a_row["close"]),
            "V": float(a_row["volume"]),
        },
        "actual": {
            "O": float(agg_bar["open"]),
            "H": float(agg_bar["high"]),
            "L": float(agg_bar["low"]),
            "C": float(agg_bar["close"]),
            "V": float(agg_bar["volume"]),
        },
        "match": all(
            abs(float(a_row[k]) - float(agg_bar[k])) <= (PRICE_TOL if k != "volume" else VOLUME_TOL)
            for k in ("open", "high", "low", "close", "volume")
        ),
    }


def summarize_tuple(
    instr: str, side: str, tf: str, df_1min: pd.DataFrame, df_cat: pd.DataFrame
) -> tuple[dict, dict, pd.DataFrame]:
    """Run the three-way comparison; return (row, example, exp_A) for one tuple."""
    if df_1min.empty or df_cat.empty:
        empty_row = {
            "instrument": instr, "side": side, "timeframe": tf,
            "src_1m_bars": int(len(df_1min)), "agg_bars": int(len(df_cat)),
            "reduction": float("nan"), "matched": 0, "mismatched": 0,
            "A_vs_cat": "SKIP", "B_vs_cat": "SKIP", "A_vs_B": "SKIP", "status": "SKIP",
        }
        return empty_row, {}, pd.DataFrame()

    exp_A = aggregate_ohlcv(df_1min, tf, drop_partial=True)
    exp_B = manual_resample(df_1min, tf)
    cmp_A = compare_frames(exp_A, df_cat)
    cmp_B = compare_frames(exp_B, df_cat)
    cmp_AB = compare_frames(exp_A, exp_B)

    ok = cmp_A["pass"] and cmp_B["pass"] and cmp_AB["pass"]
    row = {
        "instrument": instr, "side": side, "timeframe": tf,
        "src_1m_bars": int(len(df_1min)), "agg_bars": int(len(df_cat)),
        "reduction": len(df_1min) / max(len(df_cat), 1),
        "matched": cmp_A["n_match"], "mismatched": cmp_A["n_mismatch"],
        "A_vs_cat": _pf(cmp_A["pass"]),
        "B_vs_cat": _pf(cmp_B["pass"]),
        "A_vs_B":  _pf(cmp_AB["pass"]),
        "status":   "PASS" if ok else "FAIL",
    }
    example = build_example_bar(df_1min, df_cat, exp_A, tf)
    return row, example, exp_A


def build_window_index(
    df_1min: pd.DataFrame, df_cat: pd.DataFrame, timeframe: str, instr: str, side: str
) -> pd.DataFrame:
    """Per-catalog-bar window index.

    For each aggregated bar at `agg_ts`, returns a row with `window_open_expected`,
    the first/last 1-MIN timestamps inside `(window_open_expected, agg_ts]`, and the
    deltas in minutes. Vectorised via `np.searchsorted`.
    """
    n = len(df_cat)
    if n == 0 or df_1min.empty:
        return pd.DataFrame()

    agg_ts = df_cat.index
    if timeframe in TIMEFRAME_RULE_MIN:
        delta = pd.Timedelta(minutes=TIMEFRAME_RULE_MIN[timeframe])
        window_open = agg_ts - delta
        notes = np.full(n, "", dtype=object)
    else:
        prev = [pd.NaT] + list(agg_ts[:-1])
        window_open = pd.DatetimeIndex(prev, tz="UTC")
        notes = np.full(
            n,
            "variable-width bucket; window_open_expected = previous bucket label",
            dtype=object,
        )

    one_min = df_1min.index.values  # np.datetime64[ns]
    left = np.searchsorted(one_min, window_open.values, side="right")
    right = np.searchsorted(one_min, agg_ts.values, side="right") - 1

    n_bars = (right - left + 1).clip(min=0)
    valid = n_bars > 0

    left_safe = np.clip(left, 0, len(one_min) - 1)
    right_safe = np.clip(right, 0, len(one_min) - 1)
    nat = np.datetime64("NaT")
    first_arr = np.where(valid, one_min[left_safe], nat)
    last_arr = np.where(valid, one_min[right_safe], nat)

    first_ts = pd.DatetimeIndex(first_arr, tz="UTC")
    last_ts = pd.DatetimeIndex(last_arr, tz="UTC")

    delta_open_min = (first_ts - window_open).total_seconds() / 60.0
    delta_close_min = (agg_ts - last_ts).total_seconds() / 60.0
    window_span_min = (agg_ts - window_open).total_seconds() / 60.0

    return pd.DataFrame({
        "instrument": instr,
        "side": side,
        "timeframe": timeframe,
        "agg_ts": agg_ts,
        "window_open_expected": window_open,
        "first_1min_ts": first_ts,
        "last_1min_ts": last_ts,
        "n_1min_in_window": n_bars.astype(np.int64),
        "delta_open_minutes": delta_open_min,
        "delta_close_minutes": delta_close_min,
        "window_span_minutes": window_span_min,
        "notes": notes,
    })

## Loading and verifying — one `(instrument, side)` at a time

There are ~24 M 1-MIN bars across 6 `(instrument, side)` pairs. To keep memory bounded we load each pair's 1-MIN frame **once** and reuse it across all 8 timeframes before dropping it and moving on.

We also stream the per-bar window index straight to a gzipped CSV inside the loop so the script never holds all ~7.7 M window rows in memory simultaneously.

In [ ]:
if WINDOWS_CSV.exists():
    WINDOWS_CSV.unlink()
if WINDOWS_SAMPLE_CSV.exists():
    WINDOWS_SAMPLE_CSV.unlink()

rows: list[dict] = []
examples: dict = {}
sample_chunks: list[pd.DataFrame] = []
header_written = False
total_window_rows = 0

ran_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"run start: {ran_at}\n")

for instr in INSTRUMENTS:
    for side in SIDES:
        src_type = bar_type(instr, "1-MINUTE", side, SRC_SUFFIX)
        print(f"[load] {src_type}", flush=True)
        df_1min = load_bars(src_type)
        if df_1min.empty:
            print(f"  ! no 1-MIN data for {instr} {side} — skipping")
            for tf in TIMEFRAMES:
                rows.append({
                    "instrument": instr, "side": side, "timeframe": tf,
                    "src_1m_bars": 0, "agg_bars": 0, "reduction": float("nan"),
                    "matched": 0, "mismatched": 0,
                    "A_vs_cat": "SKIP", "B_vs_cat": "SKIP", "A_vs_B": "SKIP", "status": "SKIP",
                })
            continue

        for tf in TIMEFRAMES:
            dst_type = bar_type(instr, tf, side, AGG_SUFFIX)
            df_cat = load_bars(dst_type)
            print(
                f"  [verify] {instr} {side} {tf:<9} "
                f"1m={len(df_1min):>10,}  cat={len(df_cat):>9,}",
                flush=True,
            ) 
            row, ex, exp_A = summarize_tuple(instr, side, tf, df_1min, df_cat)
            rows.append(row)
            examples[(instr, side, tf)] = ex

            df_win = build_window_index(df_1min, df_cat, tf, instr, side)
            if not df_win.empty:
                df_win.to_csv(
                    WINDOWS_CSV,
                    mode="a" if header_written else "w",
                    header=not header_written,
                    index=False,
                    compression="gzip",
                    date_format="%Y-%m-%dT%H:%M:%S%z",
                )
                header_written = True
                total_window_rows += len(df_win)

                # Sample: first 1k + middle 1k + last 1k (deduped) per tuple
                n_win = len(df_win)
                idx_set = set(range(min(1000, n_win)))
                mid = n_win // 2
                idx_set.update(range(max(0, mid - 500), min(n_win, mid + 500)))
                idx_set.update(range(max(0, n_win - 1000), n_win))
                sample_chunks.append(df_win.iloc[sorted(idx_set)].copy())

            del df_cat, exp_A, df_win
        del df_1min
        gc.collect()

print(f"\nverified {len(rows)} tuples; streamed {total_window_rows:,} window rows to gz CSV")

run start: 2026-05-15T03:58:51+00:00

[load] EURUSD.FOREX_MS-1-MINUTE-ASK-EXTERNAL
  [verify] EURUSD ASK 5-MINUTE  1m= 3,923,817  cat=  785,318
  [verify] EURUSD ASK 15-MINUTE 1m= 3,923,817  cat=  262,142
  [verify] EURUSD ASK 30-MINUTE 1m= 3,923,817  cat=  131,348
  [verify] EURUSD ASK 1-HOUR    1m= 3,923,817  cat=   65,951
  [verify] EURUSD ASK 2-HOUR    1m= 3,923,817  cat=   33,252
  [verify] EURUSD ASK 1-DAY     1m= 3,923,817  cat=    3,279
  [verify] EURUSD ASK 1-WEEK    1m= 3,923,817  cat=      547
  [verify] EURUSD ASK 1-MONTH   1m= 3,923,817  cat=      125
[load] EURUSD.FOREX_MS-1-MINUTE-BID-EXTERNAL
  [verify] EURUSD BID 5-MINUTE  1m= 3,923,817  cat=  785,318
  [verify] EURUSD BID 15-MINUTE 1m= 3,923,817  cat=  262,142
  [verify] EURUSD BID 30-MINUTE 1m= 3,923,817  cat=  131,348
  [verify] EURUSD BID 1-HOUR    1m= 3,923,817  cat=   65,951
  [verify] EURUSD BID 2-HOUR    1m= 3,923,817  cat=   33,252
  [verify] EURUSD BID 1-DAY     1m= 3,923,817  cat=    3,279
  [verify] EURUSD 

## Summary

Three-way agreement column-by-column: **A vs Cat** (production aggregator vs catalog), **B vs Cat** (independent resample vs catalog), **A vs B** (production vs independent). All three must pass for `status = PASS`.

In [6]:
summary_df = pd.DataFrame(rows)
summary_df["reduction"] = summary_df["reduction"].map(
    lambda v: "n/a" if pd.isna(v) else f"{v:.2f}x"
)
summary_df["src_1m_bars"] = summary_df["src_1m_bars"].map("{:,}".format)
summary_df["agg_bars"] = summary_df["agg_bars"].map("{:,}".format)
summary_df["matched"] = summary_df["matched"].map("{:,}".format)
summary_df["mismatched"] = summary_df["mismatched"].map("{:,}".format)


def _color(val: str) -> str:
    if val == "PASS":
        return "background-color: #dafbe1; color: #116329; font-weight: 600;"
    if val == "FAIL":
        return "background-color: #ffebe9; color: #82071e; font-weight: 600;"
    if val == "SKIP":
        return "background-color: #fff8c5; color: #7d4e00; font-weight: 600;"
    return ""


summary_df.style.map(_color, subset=["A_vs_cat", "B_vs_cat", "A_vs_B", "status"])

,instrument,side,timeframe,src_1m_bars,agg_bars,reduction,matched,mismatched,A_vs_cat,B_vs_cat,A_vs_B,status
0,EURUSD,ASK,5-MINUTE,"3,923,817","785,318",5.00x,"785,318",0,PASS,PASS,PASS,PASS
1,EURUSD,ASK,15-MINUTE,"3,923,817","262,142",14.97x,"262,142",0,PASS,PASS,PASS,PASS
2,EURUSD,ASK,30-MINUTE,"3,923,817","131,348",29.87x,"131,348",0,PASS,PASS,PASS,PASS
3,EURUSD,ASK,1-HOUR,"3,923,817","65,951",59.50x,"65,951",0,PASS,PASS,PASS,PASS
4,EURUSD,ASK,2-HOUR,"3,923,817","33,252",118.00x,"33,252",0,PASS,PASS,PASS,PASS
5,EURUSD,ASK,1-DAY,"3,923,817","3,279",1196.65x,"3,279",0,PASS,PASS,PASS,PASS
6,EURUSD,ASK,1-WEEK,"3,923,817",547,7173.34x,547,0,PASS,PASS,PASS,PASS
7,EURUSD,ASK,1-MONTH,"3,923,817",125,31390.54x,125,0,PASS,PASS,PASS,PASS
8,EURUSD,BID,5-MINUTE,"3,923,817","785,318",5.00x,"785,318",0,PASS,PASS,PASS,PASS
9,EURUSD,BID,15-MINUTE,"3,923,817","262,142",14.97x,"262,142",0,PASS,PASS,PASS,PASS


In [7]:
# Rollup: one row per 1-Minute -> target timeframe, summed across all instruments and sides.
raw_rows = rows  # already a list of dicts; build a numeric DataFrame for the rollup
num_df = pd.DataFrame(raw_rows)
rollup_rows = []
for tf in TIMEFRAMES:
    sub = num_df[(num_df["timeframe"] == tf) & (num_df["status"] != "SKIP")]
    if sub.empty:
        continue
    rollup_rows.append({
        "conversion": f"1-Minute -> {tf}",
        "Σ 1-Min Bars": int(sub["src_1m_bars"].sum()),
        "Σ Agg Bars": int(sub["agg_bars"].sum()),
        "Avg Reduction": f"{sub['reduction'].mean():.2f}x",
        "Σ Matched": int(sub["matched"].sum()),
        "Σ Mismatched": int(sub["mismatched"].sum()),
        "PASS / Total": f"{int((sub['status'] == 'PASS').sum())} / {len(sub)}",
    })
rollup_df = pd.DataFrame(rollup_rows)
for c in ("Σ 1-Min Bars", "Σ Agg Bars", "Σ Matched", "Σ Mismatched"):
    rollup_df[c] = rollup_df[c].map("{:,}".format)
rollup_df

,conversion,Σ 1-Min Bars,Σ Agg Bars,Avg Reduction,Σ Matched,Σ Mismatched,PASS / Total
0,1-Minute -> 5-MINUTE,"23,554,662","4,714,268",5.00x,"4,714,268",0,6 / 6
1,1-Minute -> 15-MINUTE,"23,554,662","1,573,644",14.97x,"1,573,644",0,6 / 6
2,1-Minute -> 30-MINUTE,"23,554,662","788,488",29.87x,"788,488",0,6 / 6
3,1-Minute -> 1-HOUR,"23,554,662","395,910",59.49x,"395,910",0,6 / 6
4,1-Minute -> 2-HOUR,"23,554,662","199,616",118.00x,"199,616",0,6 / 6
5,1-Minute -> 1-DAY,"23,554,662","19,690",1196.28x,"19,690",0,6 / 6
6,1-Minute -> 1-WEEK,"23,554,662","3,286",7168.19x,"3,286",0,6 / 6
7,1-Minute -> 1-MONTH,"23,554,662",758,31076.41x,758,0,6 / 6


## Per-tuple worked example

One mid-series aggregated bar reconstructed from its underlying 1-MIN window. The "Reconstructed from 1-MIN" column is computed by `core.aggregator.aggregate_ohlcv` (Path A) and must equal the catalog-stored value within tolerance.

In [8]:
def render_example(key: tuple[str, str, str]) -> pd.DataFrame:
    ex = examples.get(key) or {}
    if not ex or "expected" not in ex:
        return pd.DataFrame()
    rule_labels = {"O": "first", "H": "max", "L": "min", "C": "last", "V": "sum"}
    rows_ex = []
    for f in ("O", "H", "L", "C", "V"):
        e = ex["expected"][f]
        a = ex["actual"][f]
        rows_ex.append({
            "Field": f,
            "Rule": rule_labels[f],
            "Reconstructed from 1-MIN": e,
            "Catalog Stored": a,
            "Diff": abs(e - a),
        })
    df = pd.DataFrame(rows_ex)
    return df


key = ("EURUSD", "ASK", "5-MINUTE")
ex = examples.get(key) or {}
if ex:
    print(f"tuple        : {key[0]} {key[1]} {key[2]}")
    print(f"agg_ts       : {ex['agg_ts']}")
    print(f"window_open  : {ex['window_start']}")
    print(f"first_1min   : {ex['first_1min_ts']}")
    print(f"last_1min    : {ex['last_1min_ts']}")
    print(f"n_1min       : {ex['n_1min']}")
    print(f"match        : {_pf(ex['match'])}")
render_example(key)

tuple        : EURUSD ASK 5-MINUTE
agg_ts       : 2020-04-01 15:20:00+00:00
window_open  : 2020-04-01 15:15:00+00:00
first_1min   : 2020-04-01 15:16:00+00:00
last_1min    : 2020-04-01 15:20:00+00:00
n_1min       : 5
match        : PASS


,Field,Rule,Reconstructed from 1-MIN,Catalog Stored,Diff
0,O,first,1.09276,1.09276,0.0
1,H,max,1.09350,1.09350,0.0
2,L,min,1.09276,1.09276,0.0
3,C,last,1.09308,1.09308,0.0
4,V,sum,594.00000,594.00000,0.0


In [9]:
# One more example from a variable-width timeframe (1-WEEK) so the (T-N, T]
# convention's behaviour around the FX weekend gap is visible in the report.
key_w = ("USDJPY", "BID", "1-WEEK")
ex_w = examples.get(key_w) or {}
if ex_w:
    print(f"tuple        : {key_w[0]} {key_w[1]} {key_w[2]}")
    print(f"agg_ts       : {ex_w['agg_ts']}")
    print(f"window_open  : {ex_w['window_start']}  <- previous bucket label (W-SUN)")
    print(f"first_1min   : {ex_w['first_1min_ts']}  <- FX market opens Sun ~21:00 UTC")
    print(f"last_1min    : {ex_w['last_1min_ts']}   <- FX market closes Fri ~21:00 UTC")
    print(f"n_1min       : {ex_w['n_1min']:,}")
    print(f"match        : {_pf(ex_w['match'])}")
render_example(key_w)

tuple        : USDJPY BID 1-WEEK
agg_ts       : 2020-04-05 00:00:00+00:00
window_open  : 2020-03-29 00:00:00+00:00  <- previous bucket label (W-SUN)
first_1min   : 2020-03-29 21:00:00+00:00  <- FX market opens Sun ~21:00 UTC
last_1min    : 2020-04-03 20:59:00+00:00   <- FX market closes Fri ~21:00 UTC
n_1min       : 7,200
match        : PASS


,Field,Rule,Reconstructed from 1-MIN,Catalog Stored,Diff
0,O,first,107.455,107.455,0.0
1,H,max,108.740,108.740,0.0
2,L,min,106.915,106.915,0.0
3,C,last,108.710,108.710,0.0
4,V,sum,1385010.000,1385010.000,0.0


## CSV emission — `(T-N, T]` window timestamps

Per aggregated bar, one row of `reports/catalog_aggregation_windows.csv.gz` records:

| Column | Meaning |
|---|---|
| `instrument`, `side`, `timeframe` | tuple identity |
| `agg_ts` | bucket label `T` (the catalog timestamp), ISO-8601 UTC |
| `window_open_expected` | `T - N` for fixed-width TFs; **previous bucket label** for 1-WEEK / 1-MONTH (variable-width) |
| `first_1min_ts` | smallest 1-MIN timestamp inside `(window_open_expected, agg_ts]` |
| `last_1min_ts` | largest 1-MIN timestamp inside that window |
| `n_1min_in_window` | count of underlying 1-MIN bars |
| `delta_open_minutes` | `first_1min_ts - window_open_expected`, in minutes |
| `delta_close_minutes` | `agg_ts - last_1min_ts`, in minutes |
| `window_span_minutes` | `agg_ts - window_open_expected`, in minutes |
| `notes` | annotated only for variable-width buckets (1-WEEK / 1-MONTH) |

The three `delta_*` columns make the gap between the aggregated bar's nominal window edges and the first/last actually-observed 1-MIN tick directly inspectable — useful for spotting FX-weekend gaps, holiday gaps, and the Sunday-evening session open.

In [10]:
# Primary CSV was streamed inside the verification loop; here we finalise the sample CSV.
if sample_chunks:
    sample_df = pd.concat(sample_chunks, ignore_index=True)
    sample_df.to_csv(
        WINDOWS_SAMPLE_CSV,
        index=False,
        date_format="%Y-%m-%dT%H:%M:%S%z",
    )
else:
    sample_df = pd.DataFrame()

primary_bytes = WINDOWS_CSV.stat().st_size if WINDOWS_CSV.exists() else 0
sample_bytes = WINDOWS_SAMPLE_CSV.stat().st_size if WINDOWS_SAMPLE_CSV.exists() else 0
print(f"primary CSV : {WINDOWS_CSV}")
print(f"   rows    : {total_window_rows:,}")
print(f"   size    : {primary_bytes / 1_000_000:.2f} MB (gzipped)")
print(f"sample  CSV : {WINDOWS_SAMPLE_CSV}")
print(f"   rows    : {len(sample_df):,}")
print(f"   size    : {sample_bytes / 1_000_000:.2f} MB")

if not sample_df.empty:
    print("\nfirst 10 rows of the sample CSV:")
sample_df.head(10) if not sample_df.empty else None

primary CSV : c:\Users\HP\OneDrive\Desktop\m-cube_version1\reports\catalog_aggregation_windows.csv.gz
   rows    : 7,695,660
   size    : 64.50 MB (gzipped)
sample  CSV : c:\Users\HP\OneDrive\Desktop\m-cube_version1\reports\catalog_aggregation_windows_sample.csv
   rows    : 112,044
   size    : 15.76 MB

first 10 rows of the sample CSV:


,instrument,side,timeframe,agg_ts,window_open_expected,first_1min_ts,last_1min_ts,n_1min_in_window,delta_open_minutes,delta_close_minutes,window_span_minutes,notes
0,EURUSD,ASK,5-MINUTE,2015-01-01 22:00:00+00:00,2015-01-01 21:55:00+00:00,2015-01-01 22:00:00+00:00,2015-01-01 22:00:00+00:00,1,5.0,0.0,5.0,
1,EURUSD,ASK,5-MINUTE,2015-01-01 22:05:00+00:00,2015-01-01 22:00:00+00:00,2015-01-01 22:01:00+00:00,2015-01-01 22:05:00+00:00,5,1.0,0.0,5.0,
2,EURUSD,ASK,5-MINUTE,2015-01-01 22:10:00+00:00,2015-01-01 22:05:00+00:00,2015-01-01 22:06:00+00:00,2015-01-01 22:10:00+00:00,5,1.0,0.0,5.0,
3,EURUSD,ASK,5-MINUTE,2015-01-01 22:15:00+00:00,2015-01-01 22:10:00+00:00,2015-01-01 22:11:00+00:00,2015-01-01 22:15:00+00:00,5,1.0,0.0,5.0,
4,EURUSD,ASK,5-MINUTE,2015-01-01 22:20:00+00:00,2015-01-01 22:15:00+00:00,2015-01-01 22:16:00+00:00,2015-01-01 22:20:00+00:00,5,1.0,0.0,5.0,
5,EURUSD,ASK,5-MINUTE,2015-01-01 22:25:00+00:00,2015-01-01 22:20:00+00:00,2015-01-01 22:21:00+00:00,2015-01-01 22:25:00+00:00,5,1.0,0.0,5.0,
6,EURUSD,ASK,5-MINUTE,2015-01-01 22:30:00+00:00,2015-01-01 22:25:00+00:00,2015-01-01 22:26:00+00:00,2015-01-01 22:30:00+00:00,5,1.0,0.0,5.0,
7,EURUSD,ASK,5-MINUTE,2015-01-01 22:35:00+00:00,2015-01-01 22:30:00+00:00,2015-01-01 22:31:00+00:00,2015-01-01 22:35:00+00:00,5,1.0,0.0,5.0,
8,EURUSD,ASK,5-MINUTE,2015-01-01 22:40:00+00:00,2015-01-01 22:35:00+00:00,2015-01-01 22:36:00+00:00,2015-01-01 22:40:00+00:00,5,1.0,0.0,5.0,
9,EURUSD,ASK,5-MINUTE,2015-01-01 22:45:00+00:00,2015-01-01 22:40:00+00:00,2015-01-01 22:41:00+00:00,2015-01-01 22:45:00+00:00,5,1.0,0.0,5.0,


## Takeaway

Every aggregated bar in the catalog passes the OHLCV invariant against both an in-process production aggregator and an independent pandas resample. The per-bar window CSV makes the `(T-N, T]` mapping queryable: read it back, group by `instrument`, `side`, `timeframe`, and the `delta_*` columns surface every gap between nominal window boundaries and the first/last observed 1-MIN tick.

Read the CSV back with:

```python
windows = pd.read_csv(
    'reports/catalog_aggregation_windows.csv.gz',
    parse_dates=['agg_ts', 'window_open_expected', 'first_1min_ts', 'last_1min_ts'],
)
```
